In [1]:
# ============================================================
# PHASE 5 - KPI Semantic Layer
# ============================================================

import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv(r"C:\Users\yipch\ecommerce-analytics-portfolio\.env")

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_USER = os.getenv("DB_USER", "root")
DB_PASS = os.getenv("DB_PASSWORD", "")
DB_NAME = os.getenv("DB_NAME", "ecommerce_analytics")

conn_str = f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine   = create_engine(conn_str, echo=False)

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM fact_events"))
    rows   = result.fetchone()[0]
    print(f"✅ Connected to MySQL")
    print(f"   fact_events rows : {rows:,}")

SQL_DIR = r"C:\Users\yipch\ecommerce-analytics-portfolio\sql"
print(f"\n✅ Setup complete. Ready to build KPI layer.")

✅ Connected to MySQL
   fact_events rows : 1,605,102

✅ Setup complete. Ready to build KPI layer.


In [2]:
# ============================================================
# STEP 2 - Create all KPI views in MySQL
# Views = saved SQL queries that act like virtual tables
# ============================================================

views = {}

# ─────────────────────────────────────────
# VIEW 1: Monthly Revenue KPIs
# Covers: Revenue, Orders, AOV, trend
# ─────────────────────────────────────────
views["vw_monthly_revenue"] = """
CREATE OR REPLACE VIEW vw_monthly_revenue AS
SELECT
    month,
    COUNT(*)                                AS total_events,
    SUM(CASE WHEN event_type = 'purchase'
             THEN 1 ELSE 0 END)             AS total_orders,
    COUNT(DISTINCT user_id)                 AS unique_users,
    COUNT(DISTINCT CASE WHEN event_type = 'purchase'
             THEN user_id END)              AS buying_users,
    ROUND(SUM(revenue), 2)                  AS total_revenue,
    ROUND(AVG(CASE WHEN event_type = 'purchase'
             THEN price END), 2)            AS avg_order_value,
    ROUND(SUM(revenue) /
          NULLIF(COUNT(DISTINCT user_id),0)
          , 2)                              AS revenue_per_user
FROM fact_events
GROUP BY month
ORDER BY month
"""

# ─────────────────────────────────────────
# VIEW 2: Conversion Funnel KPIs
# Covers: view→cart→purchase rates
# ─────────────────────────────────────────
views["vw_conversion_funnel"] = """
CREATE OR REPLACE VIEW vw_conversion_funnel AS
WITH event_counts AS (
    SELECT
        month,
        COUNT(DISTINCT CASE WHEN event_type = 'view'
              THEN user_id END)             AS viewers,
        COUNT(DISTINCT CASE WHEN event_type = 'cart'
              THEN user_id END)             AS carters,
        COUNT(DISTINCT CASE WHEN event_type = 'purchase'
              THEN user_id END)             AS buyers,
        COUNT(DISTINCT CASE WHEN event_type = 'view'
              THEN CONCAT(user_id,session_id) END) AS view_sessions,
        COUNT(DISTINCT CASE WHEN event_type = 'cart'
              THEN CONCAT(user_id,session_id) END) AS cart_sessions,
        COUNT(DISTINCT CASE WHEN event_type = 'purchase'
              THEN CONCAT(user_id,session_id) END) AS purchase_sessions
    FROM fact_events
    GROUP BY month
)
SELECT
    month,
    viewers,
    carters,
    buyers,
    ROUND(carters   * 100.0 / NULLIF(viewers,0), 2) AS view_to_cart_rate,
    ROUND(buyers    * 100.0 / NULLIF(carters,0), 2) AS cart_to_purchase_rate,
    ROUND(buyers    * 100.0 / NULLIF(viewers,0), 2) AS overall_conversion_rate,
    view_sessions,
    cart_sessions,
    purchase_sessions
FROM event_counts
ORDER BY month
"""

# ─────────────────────────────────────────
# VIEW 3: Product Performance KPIs
# Covers: revenue, conversion per product
# ─────────────────────────────────────────
views["vw_product_performance"] = """
CREATE OR REPLACE VIEW vw_product_performance AS
SELECT
    f.product_id,
    p.brand,
    p.category_l1,
    p.category_l2,
    p.avg_price,
    COUNT(*)                                        AS total_events,
    SUM(CASE WHEN f.event_type='view'
             THEN 1 ELSE 0 END)                     AS views,
    SUM(CASE WHEN f.event_type='cart'
             THEN 1 ELSE 0 END)                     AS carts,
    SUM(CASE WHEN f.event_type='purchase'
             THEN 1 ELSE 0 END)                     AS purchases,
    ROUND(SUM(f.revenue), 2)                        AS total_revenue,
    ROUND(SUM(CASE WHEN f.event_type='purchase'
             THEN 1 ELSE 0 END) * 100.0 /
          NULLIF(SUM(CASE WHEN f.event_type='view'
             THEN 1 ELSE 0 END), 0), 2)             AS view_to_purchase_rate
FROM fact_events f
JOIN dim_products p ON f.product_id = p.product_id
GROUP BY
    f.product_id, p.brand,
    p.category_l1, p.category_l2, p.avg_price
"""

# ─────────────────────────────────────────
# VIEW 4: User Retention Signals
# Covers: new vs returning, active days
# ─────────────────────────────────────────
views["vw_user_retention"] = """
CREATE OR REPLACE VIEW vw_user_retention AS
SELECT
    u.user_id,
    u.user_has_purchased,
    u.total_events,
    u.total_purchases,
    u.total_sessions,
    u.total_revenue,
    u.active_months,
    u.avg_order_value,
    DATEDIFF(u.last_seen, u.first_seen)             AS days_active,
    CASE
        WHEN u.active_months >= 2 THEN 'retained'
        WHEN u.active_months  = 1
         AND u.total_purchases > 0 THEN 'one_time_buyer'
        WHEN u.active_months  = 1
         AND u.total_purchases = 0 THEN 'one_time_visitor'
        ELSE 'unknown'
    END                                             AS retention_segment
FROM dim_users u
"""

# ─────────────────────────────────────────
# VIEW 5: Daily KPI Summary
# Covers: day-by-day operations view
# ─────────────────────────────────────────
views["vw_daily_kpis"] = """
CREATE OR REPLACE VIEW vw_daily_kpis AS
SELECT
    d.full_date,
    d.day_of_week,
    d.is_weekend,
    d.year_month,
    COUNT(*)                                        AS total_events,
    COUNT(DISTINCT f.user_id)                       AS unique_users,
    COUNT(DISTINCT f.session_id)                    AS unique_sessions,
    SUM(CASE WHEN f.event_type='view'
             THEN 1 ELSE 0 END)                     AS views,
    SUM(CASE WHEN f.event_type='cart'
             THEN 1 ELSE 0 END)                     AS carts,
    SUM(CASE WHEN f.event_type='purchase'
             THEN 1 ELSE 0 END)                     AS purchases,
    ROUND(SUM(f.revenue), 2)                        AS daily_revenue,
    ROUND(AVG(CASE WHEN f.event_type='purchase'
             THEN f.price END), 2)                  AS daily_aov
FROM fact_events f
JOIN dim_date d ON f.date_id = d.date_id
GROUP BY
    d.full_date, d.day_of_week,
    d.is_weekend, d.year_month
ORDER BY d.full_date
"""

# ─────────────────────────────────────────
# VIEW 6: Brand Performance KPIs
# ─────────────────────────────────────────
views["vw_brand_performance"] = """
CREATE OR REPLACE VIEW vw_brand_performance AS
SELECT
    p.brand,
    p.category_l1,
    COUNT(DISTINCT f.product_id)                    AS unique_products,
    SUM(CASE WHEN f.event_type='view'
             THEN 1 ELSE 0 END)                     AS total_views,
    SUM(CASE WHEN f.event_type='cart'
             THEN 1 ELSE 0 END)                     AS total_carts,
    SUM(CASE WHEN f.event_type='purchase'
             THEN 1 ELSE 0 END)                     AS total_purchases,
    ROUND(SUM(f.revenue), 2)                        AS total_revenue,
    ROUND(AVG(CASE WHEN f.event_type='purchase'
             THEN f.price END), 2)                  AS avg_selling_price,
    ROUND(SUM(CASE WHEN f.event_type='purchase'
             THEN 1 ELSE 0 END) * 100.0 /
          NULLIF(SUM(CASE WHEN f.event_type='view'
             THEN 1 ELSE 0 END),0), 2)              AS conversion_rate
FROM fact_events f
JOIN dim_products p ON f.product_id = p.product_id
WHERE p.brand != 'unknown'
GROUP BY p.brand, p.category_l1
ORDER BY total_revenue DESC
"""

# ─────────────────────────────────────────
# Execute all views
# ─────────────────────────────────────────
print("Creating KPI views in MySQL...\n")
with engine.connect() as conn:
    for view_name, view_sql in views.items():
        conn.execute(text(view_sql))
        conn.commit()
        print(f"  ✅ {view_name}")

print(f"\n✅ All {len(views)} KPI views created!")

Creating KPI views in MySQL...

  ✅ vw_monthly_revenue
  ✅ vw_conversion_funnel
  ✅ vw_product_performance
  ✅ vw_user_retention
  ✅ vw_daily_kpis
  ✅ vw_brand_performance

✅ All 6 KPI views created!


In [3]:
# ============================================================
# STEP 3 - Validate every view returns correct data
# ============================================================

print("=" * 60)
print("  KPI VIEW VALIDATION")
print("=" * 60)

validations = {
    "vw_monthly_revenue"    : "SELECT * FROM vw_monthly_revenue",
    "vw_conversion_funnel"  : "SELECT * FROM vw_conversion_funnel",
    "vw_user_retention"     : "SELECT retention_segment, COUNT(*) AS users FROM vw_user_retention GROUP BY retention_segment ORDER BY users DESC",
    "vw_daily_kpis"         : "SELECT * FROM vw_daily_kpis ORDER BY full_date LIMIT 5",
    "vw_brand_performance"  : "SELECT * FROM vw_brand_performance LIMIT 5",
    "vw_product_performance": "SELECT * FROM vw_product_performance ORDER BY total_revenue DESC LIMIT 5",
}

with engine.connect() as conn:
    for view_name, query in validations.items():
        print(f"\n📊 {view_name}:")
        result = conn.execute(text(query))
        rows   = result.fetchall()
        cols   = result.keys()
        df_out = pd.DataFrame(rows, columns=cols)
        print(df_out.to_string(index=False))

  KPI VIEW VALIDATION

📊 vw_monthly_revenue:
   month  total_events total_orders  unique_users  buying_users total_revenue avg_order_value revenue_per_user
2019-Nov        906920        12909         49981          6093    3856928.12          298.78            77.17
2019-Oct        698182        11693         49984          5626    3560026.77          304.46            71.22

📊 vw_conversion_funnel:
   month  viewers  carters  buyers view_to_cart_rate cart_to_purchase_rate overall_conversion_rate  view_sessions  cart_sessions  purchase_sessions
2019-Nov    49973    11298    6093             22.61                 53.93                   12.19         188164          24333              10975
2019-Oct    49979     5428    5626             10.86                103.65                   11.26         153186           9067               9989

📊 vw_user_retention:
retention_segment  users
 one_time_visitor  87734
   one_time_buyer  11617
         retained    307

📊 vw_daily_kpis:
 full_date da

In [5]:
# ============================================================
# STEP 4 - Master KPI Summary (the numbers that matter)
# ============================================================

print("=" * 60)
print("  MASTER KPI SUMMARY REPORT")
print("=" * 60)

with engine.connect() as conn:

    # ── Revenue KPIs ──
    r = conn.execute(text("""
        SELECT
            ROUND(SUM(revenue), 2)                      AS total_revenue,
            SUM(CASE WHEN event_type='purchase'
                     THEN 1 ELSE 0 END)                 AS total_orders,
            COUNT(DISTINCT user_id)                     AS total_users,
            COUNT(DISTINCT CASE WHEN event_type='purchase'
                     THEN user_id END)                  AS buying_users,
            ROUND(AVG(CASE WHEN event_type='purchase'
                     THEN price END), 2)                AS aov,
            ROUND(SUM(revenue) /
                  NULLIF(COUNT(DISTINCT user_id),0),2)  AS revenue_per_user
        FROM fact_events
    """))
    rev = dict(zip(r.keys(), r.fetchone()))

    # ── Funnel KPIs ──
    f = conn.execute(text("""
        SELECT * FROM vw_conversion_funnel
    """))
    funnel_df = pd.DataFrame(f.fetchall(), columns=f.keys())
    avg_conv  = funnel_df["overall_conversion_rate"].mean()
    avg_v2c   = funnel_df["view_to_cart_rate"].mean()
    avg_c2p   = funnel_df["cart_to_purchase_rate"].mean()

    # ── Retention KPIs ──
    ret = conn.execute(text("""
        SELECT retention_segment, COUNT(*) AS users
        FROM vw_user_retention
        GROUP BY retention_segment
        ORDER BY users DESC
    """))
    retention_df = pd.DataFrame(ret.fetchall(), columns=ret.keys())

   # ── Top Category ── (FIXED: revenue → total_revenue)
    cat = conn.execute(text("""
        SELECT category_l1,
               ROUND(SUM(total_revenue),2) AS total_revenue
        FROM vw_product_performance
        GROUP BY category_l1
        ORDER BY total_revenue DESC
        LIMIT 1
    """))
    top_cat = dict(zip(cat.keys(), cat.fetchone()))

print(f"""
┌─────────────────────────────────────────────┐
│           💰 REVENUE KPIs                   │
├─────────────────────────────────────────────┤
│ Total Revenue      : ${rev['total_revenue']:>10,.2f}          │
│ Total Orders       : {rev['total_orders']:>10,}          │
│ Avg Order Value    : ${rev['aov']:>10,.2f}          │
│ Total Users        : {rev['total_users']:>10,}          │
│ Buying Users       : {rev['buying_users']:>10,}          │
│ Revenue per User   : ${rev['revenue_per_user']:>10,.2f}          │
├─────────────────────────────────────────────┤
│           🎯 CONVERSION KPIs                │
├─────────────────────────────────────────────┤
│ View → Cart Rate   : {avg_v2c:>10.2f}%          │
│ Cart → Purchase    : {avg_c2p:>10.2f}%          │
│ Overall Conv. Rate : {avg_conv:>10.2f}%          │
├─────────────────────────────────────────────┤
│           👥 RETENTION KPIs                 │
├─────────────────────────────────────────────┤""")

for _, row in retention_df.iterrows():
    print(f"│ {row['retention_segment']:<20}: {row['users']:>8,} users          │")

print(f"""├─────────────────────────────────────────────┤
│           🏆 TOP CATEGORY                   │
├─────────────────────────────────────────────┤
│ {top_cat['category_l1']:<20}: ${top_cat['total_revenue']:>10,.2f}          │
└─────────────────────────────────────────────┘""")

  MASTER KPI SUMMARY REPORT

┌─────────────────────────────────────────────┐
│           💰 REVENUE KPIs                   │
├─────────────────────────────────────────────┤
│ Total Revenue      : $7,416,954.89          │
│ Total Orders       :     24,602          │
│ Avg Order Value    : $    301.48          │
│ Total Users        :     99,658          │
│ Buying Users       :     11,698          │
│ Revenue per User   : $     74.42          │
├─────────────────────────────────────────────┤
│           🎯 CONVERSION KPIs                │
├─────────────────────────────────────────────┤
│ View → Cart Rate   :      16.73%          │
│ Cart → Purchase    :      78.79%          │
│ Overall Conv. Rate :      11.72%          │
├─────────────────────────────────────────────┤
│           👥 RETENTION KPIs                 │
├─────────────────────────────────────────────┤
│ one_time_visitor    :   87,734 users          │
│ one_time_buyer      :   11,617 users          │
│ retained            :      

In [6]:
# ============================================================
# STEP 5 - Save all views as SQL files for portfolio
# ============================================================

for view_name, view_sql in views.items():
    path = os.path.join(SQL_DIR, f"{view_name}.sql")
    with open(path, "w") as f:
        f.write(f"USE ecommerce_analytics;\n\n")
        f.write(view_sql.strip())
    print(f"✅ Saved: sql/{view_name}.sql")

print(f"\n🎯 Your sql/ folder now has {len(views)} KPI view files")
print("   These are portfolio-ready SQL samples for interviews!")

✅ Saved: sql/vw_monthly_revenue.sql
✅ Saved: sql/vw_conversion_funnel.sql
✅ Saved: sql/vw_product_performance.sql
✅ Saved: sql/vw_user_retention.sql
✅ Saved: sql/vw_daily_kpis.sql
✅ Saved: sql/vw_brand_performance.sql

🎯 Your sql/ folder now has 6 KPI view files
   These are portfolio-ready SQL samples for interviews!
